# DIE05: SQLite Local Database Management

**SQLite** es un motor de base de datos relacional completo que no requiere un servidor independiente; guarda toda la base de datos en un solo archivo físico (ej. `.db`). 

Es la herramienta perfecta para analistas de datos cuando los archivos CSV son insuficientes y se necesita integridad relacional (tablas conectadas) sin la complejidad de configurar un servidor MySQL o PostgreSQL.

In [1]:
import sqlite3
import pandas as pd

# Conectar a la base de datos (crea el archivo físicamente si no existe en la carpeta)
db_path = "DIE05_concesionario.db"
connection = sqlite3.connect(db_path)
cursor = connection.cursor()

# --- DDL (Data Definition Language): Creación de Tablas ---

cursor.execute("""
CREATE TABLE IF NOT EXISTS marcas(
    id_marca INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    pais_origen TEXT
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS clientes(
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nombres TEXT NOT NULL,
    apellidos TEXT NOT NULL,
    telefono TEXT,
    correo TEXT
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS vendedores(
    id_vendedor INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    cargo TEXT,
    salario REAL
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS vehiculos(
    id_vehiculo INTEGER PRIMARY KEY AUTOINCREMENT,
    modelo TEXT NOT NULL,
    anio INTEGER,
    color TEXT,
    precio REAL,
    stock INTEGER,
    id_marca INTEGER,
    FOREIGN KEY(id_marca) REFERENCES marcas(id_marca)
)""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS ventas(
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha TEXT,
    id_cliente INTEGER,
    id_vendedor INTEGER,
    id_vehiculo INTEGER,
    cantidad INTEGER,
    total REAL,
    FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente),
    FOREIGN KEY(id_vendedor) REFERENCES vendedores(id_vendedor),
    FOREIGN KEY(id_vehiculo) REFERENCES vehiculos(id_vehiculo)
)""")

print(f"Esquema de base de datos creado exitosamente en {db_path}.")

Esquema de base de datos creado exitosamente en DIE05_concesionario.db.


## 1. Bulk Data Insertion
El método `.executemany()` es altamente eficiente para insertar múltiples filas a la vez utilizando listas de tuplas. Los signos de interrogación `?` actúan como *placeholders* seguros.

In [2]:
# --- DML (Data Manipulation Language): Inserción de Datos ---

cursor.executemany("INSERT INTO marcas(nombre, pais_origen) VALUES (?, ?)", [
    ("Toyota", "Japón"), ("Honda", "Japón"), ("Ford", "Estados Unidos"),
    ("Chevrolet", "Estados Unidos"), ("Hyundai", "Corea del Sur")
])

cursor.executemany("INSERT INTO clientes(nombres, apellidos, telefono, correo) VALUES (?, ?, ?, ?)", [
    ("Juan", "Pérez", "999111222", "juan@gmail.com"),
    ("María", "López", "988777666", "maria@gmail.com"),
    ("Carlos", "Ramírez", "977555444", "carlos@gmail.com")
])

cursor.executemany("INSERT INTO vendedores(nombre, cargo, salario) VALUES (?, ?, ?)", [
    ("Pedro Gómez", "Asesor", 2500), ("Lucía Díaz", "Asesor", 2600), ("José Herrera", "Supervisor", 3500)
])

cursor.executemany("INSERT INTO vehiculos(modelo, anio, color, precio, stock, id_marca) VALUES (?, ?, ?, ?, ?, ?)", [
    ("Corolla", 2024, "Blanco", 23000, 5, 1),
    ("Civic", 2024, "Negro", 25000, 4, 2),
    ("Ranger", 2023, "Azul", 38000, 3, 3)
])

cursor.executemany("INSERT INTO ventas(fecha, id_cliente, id_vendedor, id_vehiculo, cantidad, total) VALUES (?, ?, ?, ?, ?, ?)", [
    ("2025-01-10", 1, 1, 1, 1, 23000),
    ("2025-01-15", 2, 2, 2, 1, 25000)
])

# Confirmar transacciones
connection.commit()
print("Datos insertados y confirmados (commit).")

Datos insertados y confirmados (commit).


## 2. Querying Data into Pandas
Ahora que nuestra base de datos relacional local está lista, podemos tratarla exactamente igual que a un servidor remoto grande, usando nuestra conexión directamente con Pandas.

In [3]:
# Leemos los datos de SQLite directamente a un DataFrame de Pandas
query_reporte = """
SELECT v.fecha, c.nombres AS cliente, veh.modelo AS auto, v.total
FROM ventas v
JOIN clientes c ON v.id_cliente = c.id_cliente
JOIN vehiculos veh ON v.id_vehiculo = veh.id_vehiculo;
"""

df_ventas = pd.read_sql_query(query_reporte, con=connection)

print("--- Reporte de Ventas desde SQLite ---")
display(df_ventas)

# Buena práctica: Siempre cerrar la conexión al terminar
connection.close()
print("\nConexión a SQLite cerrada de forma segura.")

--- Reporte de Ventas desde SQLite ---


,fecha,cliente,auto,total
0,2025-01-10,Juan,Corolla,23000.0
1,2025-01-15,María,Civic,25000.0



Conexión a SQLite cerrada de forma segura.
